In [6]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END 

class State(TypedDict):
    amount: float
    decision:str 
    
def route(state: State) -> Literal["auto_approve", "human_review"]:
    if state['amount'] < 100:
        return "auto_approve"
    else:
        return "human_review"

def auto_approve(state):
    return {"decision": "approved automatically"}

def human_review(state):
    return {"decision": "needs human review"}

g = StateGraph(State)
g.add_node("auto_approve", auto_approve)
g.add_node("human_review", human_review)
g.add_conditional_edges(START, route)
g.add_edge("auto_approve", END)
g.add_edge("human_review", END)
graph = g.compile() 

In [7]:
print(graph.invoke({"amount": 50}))
print(graph.invoke({"amount": 150}))

{'amount': 50, 'decision': 'approved automatically'}
{'amount': 150, 'decision': 'needs human review'}


In [8]:
assert route({"amount": 99.99, "decision": ""}) == "auto_approve"
assert route({"amount": 100, "decision": ""}) == "human_review"
print("router tests passed")

router tests passed


Next example :- When one node updates state and choose new pathm 

In [18]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command 

class State(TypedDict):
    amount: float
    decision:str

def decide(state: State) -> Command[Literal["auto_approve", "human_review"]]:
    # one node does BOTH: update state and choose the next node
    if state['amount'] < 100:
        return Command(update={"decision": "small amount"}, goto="auto_approve")
    else:
        return Command(update={"decision": "large amount"}, goto="human_review")
    

def auto_approve(state):
    return {"decision": state["decision"] + " -> approved"}

def human_review(state):
    return {"decision": state["decision"] + " -> needs human review"}

g = StateGraph(State)
g.add_node("decide", decide)
g.add_node("auto_approve", auto_approve)
g.add_node("human_review", human_review)
g.add_edge(START, "decide")
g.add_edge("auto_approve", END)
g.add_edge("human_review", END)
graph = g.compile()

In [19]:
print(graph.invoke({"amount": 50, "decision": ""}))
print(graph.invoke({"amount": 150, "decision": ""}))

{'amount': 50, 'decision': 'small amount -> approved'}
{'amount': 150, 'decision': 'large amount -> needs human review'}
